# Multi-View 3D Mouse Pose, End-to-End Pipeline

Consolidated notebook covering:

1. **Setup**, imports & paths
2. **Preprocessing**, CLAHE etc. for IR top-down footage
3. **Inference**, run SuperAnimal on every view variant (with & without adaptation)
4. **Comparison**, detection coverage plots side-by-side
5. **Choose best top**, pick which top-view variant to use downstream
6. **Frame visualization**, top/side/front overlay on a synced frame
7. **Keypoint mapping & export**, bridge TopViewMouse and Quadruped naming, then save the 2D state

3D reconstruction (calibration + triangulation) lives in the following notebooks
(`01_lockbox_calibration`, `02_triangulate_and_render`).

## 1. Imports & paths

In [1]:
# Standard libs
import os, gc, json, shutil, warnings
from pathlib import Path

# Numerics / data
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D  # noqa

# Image / video
import cv2

# DLC + Aniposelib
import deeplabcut
from deeplabcut.modelzoo.video_inference import video_inference_superanimal
from aniposelib.cameras import Camera, CameraGroup

warnings.filterwarnings('ignore')
print(f"DLC version: {deeplabcut.__version__}")
print(f"OpenCV:      {cv2.__version__}")

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


/home/kenny/HTCV/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DLC version: 3.0.0rc13
OpenCV:      4.11.0


In [ ]:
# Paths
DATA_ROOT   = Path("../data/validation/scene1").resolve()
OUTPUT_ROOT = Path("../data/multiview_output/scene1").resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RAW_TOP    = DATA_ROOT / "2021-06-18_07-28-45_segment1_mouse291_combined_top-down-view.avi"
RAW_TOP_T2 = DATA_ROOT / "2021-06-18_07-28-45_segment1_mouse291_combined_top-down-view_t2.avi"
CLAHE_TOP  = DATA_ROOT / "top_clahe.mp4"
RAW_SIDE   = DATA_ROOT / "2021-06-18_07-28-45_segment1_mouse291_combined_side-view.avi"
RAW_FRONT  = DATA_ROOT / "2021-06-18_07-28-45_segment1_mouse291_combined_front-view.avi"

# Every variant we want to compare.
# `folder` is the subfolder in OUTPUT_ROOT (multiple keys can share a folder).
# `after_adapt` chooses which h5 to load: True = post-adaptation, False = pre-adaptation.
# `adapt` controls whether the inference cell should run adaptation if not yet done.
INFERENCE_CONFIGS = [
    # key             video        sa_model                        backbone     folder    adapt  after_adapt
    ('top_noadapt',   RAW_TOP,     'superanimal_topviewmouse',     'hrnet_w32', 'top',    True,  False),
    ('top',           RAW_TOP,     'superanimal_topviewmouse',     'hrnet_w32', 'top',    True,  True ),
    ('top2',          RAW_TOP_T2,  'superanimal_quadruped',        'hrnet_w32', 'top2',   True,  True ),
    ('top3',          CLAHE_TOP,   'superanimal_topviewmouse',     'hrnet_w32', 'top3',   True,  True ),
    ('side',          RAW_SIDE,    'superanimal_quadruped',        'hrnet_w32', 'side',   True,  True ),
    ('front',         RAW_FRONT,   'superanimal_quadruped',        'hrnet_w32', 'front',  True,  True ),
]
VIEW_OUT = {}
for key, video, sa, mb, folder, adapt, after_adapt in INFERENCE_CONFIGS:
    VIEW_OUT[key] = OUTPUT_ROOT / folder

for d in set(VIEW_OUT.values()):
    d.mkdir(parents=True, exist_ok=True)

# Sanity check
print(f"{'key':<14} {'folder':<8} {'video':<60} {'after_adapt'}")
print("─"*100)
for key, video, sa, mb, folder, adapt, after_adapt in INFERENCE_CONFIGS:
    print(f"  {key:<11} {folder:<8} {video.name:<60} {after_adapt}")

key            folder   video                                                        after_adapt
────────────────────────────────────────────────────────────────────────────────────────────────────
   top_noadapt top      2021-06-18_07-28-45_segment1_mouse291_combined_top-down-view.avi False
   top         top      2021-06-18_07-28-45_segment1_mouse291_combined_top-down-view.avi True
   top2        top2     2021-06-18_07-28-45_segment1_mouse291_combined_top-down-view_t2.avi True
   top3        top3     top_clahe.mp4                                                True
   side        side     2021-06-18_07-28-45_segment1_mouse291_combined_side-view.avi True
   front       front    2021-06-18_07-28-45_segment1_mouse291_combined_front-view.avi True


## 2. Preprocessing the IR top-down video

IR illumination produces low-contrast frames very different from SuperAnimal's training distribution. CLAHE typically helps a lot.

In [3]:
def to_gray(f):     return cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) if f.ndim == 3 else f
def gray_to_bgr(g): return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)

def variant_clahe(frame, clip=3.0, tile=8):
    g = to_gray(frame)
    return gray_to_bgr(cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(g))

def variant_gamma(frame, gamma=1.5):
    tbl = np.array([((i/255.0)**(1/gamma))*255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(frame, tbl)

def variant_normalize(frame):
    return gray_to_bgr(cv2.normalize(to_gray(frame), None, 0, 255, cv2.NORM_MINMAX))

VARIANTS = {
    'raw':                lambda f: f,
    'clahe':              variant_clahe,
    'clahe_strong':       lambda f: variant_clahe(f, clip=5.0),
    'invert':             lambda f: 255 - f,
    'clahe_then_invert':  lambda f: 255 - variant_clahe(f),
    'gamma_1.5':          lambda f: variant_gamma(f, 1.5),
    'gamma_0.7':          lambda f: variant_gamma(f, 0.7),
    'normalize':          variant_normalize,
    'normalize_invert':   lambda f: 255 - variant_normalize(f),
}

# Show variants on a sample frame
def get_frame(path, idx):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    _, f = cap.read(); cap.release(); return f

sample = get_frame(RAW_TOP, 100)
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for ax, (name, fn) in zip(axes.ravel(), VARIANTS.items()):
    ax.imshow(cv2.cvtColor(fn(sample.copy()), cv2.COLOR_BGR2RGB))
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Write the CLAHE-preprocessed top video if it doesn't exist yet
CHOSEN_VARIANT = 'clahe'   # ← edit based on what looked best above

if CLAHE_TOP.exists():
    print(f" Preprocessed video already exists: {CLAHE_TOP}")
else:
    fn = VARIANTS[CHOSEN_VARIANT]
    cap = cv2.VideoCapture(str(RAW_TOP))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(CLAHE_TOP), fourcc, fps, (w, h))
    print(f'Writing {n} frames -> {CLAHE_TOP}')
    for i in range(n):
        ok, frame = cap.read()
        if not ok: break
        writer.write(fn(frame))
        if i % 500 == 0: print(f'  {i}/{n}')
    cap.release(); writer.release()
    print('Done')

Preprocessed video already exists: /home/kenny/HTCV/data/validation/scene1/top_clahe.mp4


## 3. Inference on every view variant

Each variant lands in its own output folder. Skip any that are already done.

In [5]:
# Common inference settings
INFER_KW = dict(
    detector_name="fasterrcnn_resnet50_fpn_v2",
    scale_list=[],
    batch_size=16,
    detector_batch_size=4,
    pcutoff=0.1,
    max_individuals=1,
    bbox_threshold=0.3,
    create_labeled_video=True,
    plot_bboxes=True,
)
ADAPT_KW = dict(
    video_adapt=True,
    video_adapt_batch_size=1,
    adapt_iterations=1000,
    pseudo_threshold=0.5,
    pose_epochs=4,
    detector_epochs=4,
)

def flush_vram():
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for key, video, sa, mb, folder, adapt, after_adapt in INFERENCE_CONFIGS:
    # Skip the "noadapt" view if its sibling adapt-run was already done —
    # both share the same folder and the before_adapt h5 is created automatically
    if not after_adapt:
        print(f"  {key:18s}: shares folder with adapt run — h5 already exists")
        continue

    out_dir = VIEW_OUT[key]
    # check for the AFTER-adapt h5 specifically
    if any('snapshot' in p.name for p in out_dir.glob("*.h5")):
        print(f"  {key:18s}: already processed")
        continue
    if not video.exists():
        print(f"✗  {key:18s}: source video missing — skipping")
        continue

    flush_vram()
    print(f"\n  {key}: {sa} / {mb} | adapt={adapt}")
    kwargs = dict(INFER_KW)
    if adapt:
        kwargs.update(ADAPT_KW)
    else:
        kwargs['video_adapt'] = False

    video_inference_superanimal(
        videos=[str(video)],
        superanimal_name=sa,
        model_name=mb,
        videotype=video.suffix,
        dest_folder=str(out_dir),
        **kwargs,
    )
    flush_vram()

print("\n All inference done")

  top_noadapt       : shares folder with adapt run — h5 already exists
  top               : already processed
  top2              : already processed
  top3              : already processed
  side              : already processed
  front             : already processed

All inference done


## 4. Compare detection coverage across variants

Two plots:
- **Coverage timeline**, fraction of keypoints detected per frame, one line per variant
- **Keypoint trajectories per view**, x/y over time, with the per-keypoint detail you wanted

In [6]:
def load_predictions(key):
    """Return DataFrame with (bodypart, coord) columns, or None.
    Picks before- or after-adaptation h5 based on the INFERENCE_CONFIGS entry."""
    cfg = next((c for c in INFERENCE_CONFIGS if c[0] == key), None)
    if cfg is None: return None
    after_adapt = cfg[6]

    folder = VIEW_OUT[key]
    h5s = list(folder.glob("*.h5"))
    if not h5s: return None

    # before_adapt files have e.g. "_hrnet_w32_fasterrcnn_..." in the name (no "snapshot")
    # after_adapt files have e.g. "_snapshot-hrnet_w32-004_snapshot-fasterrcnn-..." (with "snapshot")
    if after_adapt:
        h5s = [h for h in h5s if 'snapshot' in h.name]
    else:
        h5s = [h for h in h5s if 'snapshot' not in h.name]
    if not h5s: return None

    df = pd.read_hdf(h5s[0])
    while df.columns.nlevels > 2:
        df = df.droplevel(0, axis=1)
    return df

In [7]:
all_preds = {key: load_predictions(key) for key, *_ in INFERENCE_CONFIGS}

for k, df in all_preds.items():
    status = f"{len(df)} frames, {len(df.columns.get_level_values(0).unique())} kps" if df is not None else "— no h5"
    print(f"  {k:18s}: {status}")

  top_noadapt       : 8190 frames, 27 kps
  top               : 8190 frames, 27 kps
  top2              : 8190 frames, 39 kps
  top3              : 8190 frames, 27 kps
  side              : 8190 frames, 39 kps
  front             : 8190 frames, 39 kps


In [8]:
# Coverage timeline — fraction of keypoints detected per frame, one line per variant
CONF = 0.4
fig, ax = plt.subplots(figsize=(14, 5))

for key, df in all_preds.items():
    if df is None: continue
    likelihoods = df.xs('likelihood', axis=1, level=1)
    coverage = (likelihoods > CONF).mean(axis=1) * 100   # % of kps detected per frame
    mean_cov = coverage.mean()
    ax.plot(coverage.rolling(30, min_periods=1).mean(),
            label=f'{key} (mean {mean_cov:.0f}%)',
            linewidth=1.4, alpha=0.85)

ax.set_xlabel('frame')
ax.set_ylabel('% of keypoints detected (rolling 30-frame mean)')
ax.set_title(f'Detection coverage per variant (likelihood > {CONF})')
ax.legend(loc='center right', fontsize=9, framealpha=0.9)
ax.set_ylim(0, 105); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [9]:
# Per-view keypoint trajectories (x and y over time, first 6 keypoints)
PCUTOFF_PLOT = 0.3
N_BP_PLOT    = 6

variants_to_plot = [k for k, df in all_preds.items() if df is not None]
n_rows = len(variants_to_plot)

fig, axes = plt.subplots(n_rows, 2, figsize=(16, 2.5 * n_rows), sharex='col')
if n_rows == 1:
    axes = axes.reshape(1, 2)

for r, key in enumerate(variants_to_plot):
    df = all_preds[key]
    bps = df.columns.get_level_values(0).unique().tolist()[:N_BP_PLOT]
    for bp in bps:
        x = df[bp]['x']; y = df[bp]['y']; l = df[bp]['likelihood']
        x = x.where(l > PCUTOFF_PLOT); y = y.where(l > PCUTOFF_PLOT)
        axes[r, 0].plot(x.values, label=bp, lw=0.7)
        axes[r, 1].plot(y.values, label=bp, lw=0.7)
    axes[r, 0].set_ylabel(f'{key}\nX (px)', fontsize=9)
    axes[r, 1].set_ylabel('Y (px)', fontsize=9)
    axes[r, 0].legend(ncol=3, fontsize=6, loc='lower right')

axes[-1, 0].set_xlabel('frame'); axes[-1, 1].set_xlabel('frame')
fig.suptitle('Keypoint trajectories per view variant', fontsize=12)
plt.tight_layout(); plt.show()

## 5. Pick the best top-view variant for 3D

Based on the coverage plot above, set `TOP_KEY` to whichever variant has the best detection rate.

In [10]:
TOP_KEY = 'top'

VIEWS_3D = {
    'top':   TOP_KEY,
    'side':  'side',
    'front': 'front',
}

VIDEO_FOR_VIEW = {
    v: next(c[1] for c in INFERENCE_CONFIGS if c[0] == k)
    for v, k in VIEWS_3D.items()
}

print("Using for 3D:")
for v, k in VIEWS_3D.items():
    print(f"  {v:5s} ← {k:13s}  ({VIDEO_FOR_VIEW[v].name})")

Using for 3D:
  top   ← top            (2021-06-18_07-28-45_segment1_mouse291_combined_top-down-view.avi)
  side  ← side           (2021-06-18_07-28-45_segment1_mouse291_combined_side-view.avi)
  front ← front          (2021-06-18_07-28-45_segment1_mouse291_combined_front-view.avi)


## 6. Side-by-side frame visualization

In [11]:
def plot_keypoints(ax, df, frame_idx, pcutoff=0.5):
    bps = df.columns.get_level_values(0).unique()
    row = df.iloc[frame_idx]
    for bp in bps:
        l = row[bp]['likelihood']
        if l < pcutoff: continue
        x, y = row[bp]['x'], row[bp]['y']
        ax.scatter(x, y, s=25, c=[plt.cm.RdYlGn(l)], edgecolors='k', linewidths=0.4, zorder=5)
        ax.text(x+3, y-3, bp, fontsize=4, color='white',
                bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.5))

# ← Edit frame index after looking at the coverage plot for a frame with good coverage in all views
FRAME_IDX = 500

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, view in zip(axes, ['top', 'side', 'front']):
    df = all_preds[VIEWS_3D[view]]
    img = get_frame(VIDEO_FOR_VIEW[view], FRAME_IDX)
    if img is None or df is None:
        ax.set_title(f'{view}: missing'); continue
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plot_keypoints(ax, df, FRAME_IDX)
    ax.set_title(f'{view} ({VIEWS_3D[view]}) — frame {FRAME_IDX}')
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import pickle

# ── Keypoint mapping ──────────────────────────────────────────────────────────
bp_by_view = {v: all_preds[VIEWS_3D[v]].columns.get_level_values(0).unique().tolist()
              for v in ['top', 'side', 'front']}

KEYPOINT_MAP = {
    # canonical          top                 side                 front
    'nose':              ('nose',            'nose',              'nose'),
    'left_eye':          ('left_eye',        'left_eye',          'left_eye'),
    'right_eye':         ('right_eye',       'right_eye',         'right_eye'),
    'left_ear':          ('left_ear',        'left_earbase',      'left_earbase'),
    'right_ear':         ('right_ear',       'right_earbase',     'right_earbase'),
    'left_ear_tip':      ('left_ear_tip',    'left_earend',       'left_earend'),
    'right_ear_tip':     ('right_ear_tip',   'right_earend',      'right_earend'),
    'head_mid':          ('head_midpoint',   'neck_end',          'neck_end'),
    'neck':              ('neck',            'neck_base',         'neck_base'),
    'mid_back':          ('mid_back',        'back_middle',       'back_middle'),
    'tail_base':         ('tail_base',       'tail_base',         'tail_base'),
    'tail_end':          ('tail_end',        'tail_end',          'tail_end'),
    'left_shoulder':     ('left_shoulder',   'front_left_thai',   'front_left_thai'),
    'right_shoulder':    ('right_shoulder',  'front_right_thai',  'front_right_thai'),
    'left_hip':          ('left_hip',        'back_left_thai',    'back_left_thai'),
    'right_hip':         ('right_hip',       'back_right_thai',   'back_right_thai'),
    'front_left_paw':    (None,              'front_left_paw',    'front_left_paw'),
    'front_right_paw':   (None,              'front_right_paw',   'front_right_paw'),
    'back_left_paw':     (None,              'back_left_paw',     'back_left_paw'),
    'back_right_paw':    (None,              'back_right_paw',    'back_right_paw'),
}

# Build valid_kps (all keypoints that map cleanly across views)
valid_kps = []
for canon, (t, s, f) in KEYPOINT_MAP.items():
    ok_t = (t is None) or (t in bp_by_view['top'])
    ok_s = (s is None) or (s in bp_by_view['side'])
    ok_f = (f is None) or (f in bp_by_view['front'])
    if ok_t and ok_s and ok_f:
        valid_kps.append(canon)
n_kps = len(valid_kps)
print(f'{n_kps} valid keypoints: {valid_kps}')

# Skeleton definition
SKELETON = [
    ('nose', 'left_eye'),  ('nose', 'right_eye'),
    ('left_eye', 'left_ear'), ('right_eye', 'right_ear'),
    ('left_ear', 'left_ear_tip'), ('right_ear', 'right_ear_tip'),
    ('left_ear', 'head_mid'), ('right_ear', 'head_mid'),
    ('head_mid', 'neck'),  ('neck', 'mid_back'),
    ('mid_back', 'tail_base'), ('tail_base', 'tail_end'),
    ('neck', 'left_shoulder'),  ('neck', 'right_shoulder'),
    ('tail_base', 'left_hip'),  ('tail_base', 'right_hip'),
    ('left_shoulder', 'front_left_paw'),
    ('right_shoulder', 'front_right_paw'),
    ('left_hip', 'back_left_paw'),
    ('right_hip', 'back_right_paw'),
]

# Export
EXPORT_OUT = Path("../data/pipeline_export/scene1").resolve()
EXPORT_OUT.mkdir(parents=True, exist_ok=True)

state_2d = {
    'view_predictions': {v: all_preds[VIEWS_3D[v]] for v in ['top', 'side', 'front']},
    'valid_kps':        valid_kps,
    'KEYPOINT_MAP':     KEYPOINT_MAP,
    'SKELETON':         SKELETON,
    'VIEWS_3D':         VIEWS_3D,
    'VIDEO_FOR_VIEW':   {k: str(v) for k, v in VIDEO_FOR_VIEW.items()},
    'n_kps':            n_kps,
}
with open(EXPORT_OUT / '2d_state.pkl', 'wb') as f:
    pickle.dump(state_2d, f)
print(f'2D state saved -> {EXPORT_OUT / "2d_state.pkl"}')

20 valid keypoints: ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear', 'left_ear_tip', 'right_ear_tip', 'head_mid', 'neck', 'mid_back', 'tail_base', 'tail_end', 'left_shoulder', 'right_shoulder', 'left_hip', 'right_hip', 'front_left_paw', 'front_right_paw', 'back_left_paw', 'back_right_paw']
2D state saved -> /home/kenny/HTCV/data/pipeline_export/scene1/2d_state.pkl
